# Business Entity Resolution — full test run

Runs the modular pipeline (`src/entity_resolution`) end to end on the **full test set**
and writes the submission files:

1. **Setup** — locate `src/`, check CUDA, show which blockers and backends are active
2. **Blocking check** *(optional)* — recall/cost of each blocker on the full training pool
3. **Train** — sample S1, block, build features, XGBoost + F₀.₅ threshold (or load a saved model)
4. **Predict** — country-by-country blocking + scoring on the whole test set
5. **Validate** — the official `utils/validate_submission.py` checks
6. **Results** — per-country match statistics, output preview, stage timings
7. **Package** — `<team_name>_submission.zip` in the structure the challenge README requires

Outputs: `student_resource/output/matching_results.tsv` (upload this to the portal) and
`student_resource/output/candidate_pairs.tsv`.

**Short-lived sessions:** every expensive stage is checkpointed. If the session dies, start it
again and **Run All** — finished stages (data sample, candidates, features, each XGBoost grid
config, hard negatives, each test-prediction shard of 100K entities) load from disk and the run
continues where it stopped. Checkpoints live in `student_resource/.checkpoints/`.

Built for a large GPU box (e.g. Lightning AI A100, ~180 GB RAM). The first run re-preprocesses all
six source files into the parquet cache (transliteration changed the cache key); later runs reuse it.
Run the cells top to bottom.

In [ ]:
# Lightning AI, first time only: install dependencies, then restart the kernel.
# %pip install -r requirements.txt

import os
import sys
import time
from contextlib import contextmanager
from pathlib import Path

# Find src/ whether Jupyter was started from the repo root, student_resource/,
# code/business_entity_resolution/ or src/.
SRC = None
for base in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    for rel in ("student_resource/code/business_entity_resolution/src",
                "code/business_entity_resolution/src", "business_entity_resolution/src", "src", "."):
        if (base / rel / "entity_resolution" / "pipeline.py").exists():
            SRC = (base / rel).resolve()
            break
    if SRC:
        break
if SRC is None:
    raise FileNotFoundError("Could not find src/entity_resolution — start Jupyter inside the repo.")
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
print(f"src: {SRC}")

TIMINGS = {}

@contextmanager
def timed(label):
    t0 = time.time()
    yield
    TIMINGS[label] = time.time() - t0
    print(f"\n[Timing] {label}: {TIMINGS[label] / 60:.1f} min")

## Settings

In [ ]:
TRAIN_MODEL = True        # False: skip training and load src/models/xgb_model.pkl
SAMPLE_SIZE = 50_000      # S1 training entities (train + 10% validation split)
CLEAR_CACHE = False       # True: delete the parquet/embedding cache first
RESUME = True             # continue from saved stages; False recomputes this run's stages
CLEAR_CHECKPOINTS = False # True: delete every saved checkpoint first

TEAM_NAME = "your_team_name"  # REQUIRED: your registered team name; the zip step refuses the placeholder

RUN_BLOCKING_EVAL = False # optional pre-check on the full training pool (US + India)
EVAL_COUNTRIES = ["us", "india"]
EVAL_N_QUERIES = 5_000

## 1. Setup and environment checks

Fails here, before any data is loaded, if CUDA is unavailable. Shows which optional
libraries are installed, the TF-IDF retrieval backend, and the blocking configuration.

In [ ]:
from entity_resolution import config, runtime
from entity_resolution.blocking import HNSW_AVAILABLE
from entity_resolution.blocking.tfidf import _backend
from entity_resolution.cache import clear_cache
from entity_resolution.checkpoint import clear_checkpoints, list_runs

runtime.configure_stdio()
runtime.report_optional_dependencies()
runtime.check_cuda()
_backend()  # prints GPU (CuPy) or CPU (SciPy) for TF-IDF retrieval

if CLEAR_CACHE:
    clear_cache()
if CLEAR_CHECKPOINTS:
    clear_checkpoints()

runs = list_runs()
print(f"Checkpoints: {config.CHECKPOINT_DIR}")
for path, kind, created, stages in runs:
    print(f"  {kind:8s} {os.path.basename(path)}  (started {created}) — {len(stages)} stage(s): "
          + ", ".join(stages[:8]) + (" ..." if len(stages) > 8 else ""))
if not runs:
    print("  none yet — this is a fresh start")

print(f"""
Data         : {config.BASE_DIR}/dataset
Output       : {config.OUTPUT_DIR}
Model        : {config.MODEL_PATH}
Cache        : {config.CACHE_DIR}
Workers      : {config.N_WORKERS} (CPUs: {os.cpu_count()})

Blocking     : char TF-IDF top-{config.TFIDF_TOP_K} {'on' if config.USE_CHAR_TFIDF else 'off'} | \
word TF-IDF top-{config.WORD_TFIDF_TOP_K} {'on' if config.USE_WORD_TFIDF else 'off'} | \
name TF-IDF {'top-' + str(config.NAME_TFIDF_TOP_K) if config.NAME_TFIDF_TOP_K else 'off'}
               HNSW {'on' if config.USE_HNSW and HNSW_AVAILABLE else 'off'} | \
key indexes {'on' if config.USE_KEY_INDEXES else 'off'} | LSH {'on' if config.USE_LSH else 'off'} | \
max_df={config.TFIDF_MAX_DF}
Pre-filter   : cap {config.PREFILTER_MAX_CANDIDATES}, min score {config.PREFILTER_MIN_SCORE}
Transliterate: {config.TRANSLITERATE}
""")
if config.USE_HNSW and not HNSW_AVAILABLE:
    print("NOTE: HNSW is enabled but sentence-transformers/hnswlib are not installed — it will be skipped.")

## 2. Blocking check on the full training pool *(optional)*

Skipped unless `RUN_BLOCKING_EVAL = True`. Blocks `EVAL_N_QUERIES` training entities per country
against that country's **full** S2+S3 training pool and prints each blocker's recall, cost and
unique contribution. Use it to check the blocking settings before spending time on the test run.
France has no training data, so it can't be evaluated here.

In [ ]:
if RUN_BLOCKING_EVAL:
    from entity_resolution.blocking_eval import run as eval_blocking
    for country in EVAL_COUNTRIES:
        with timed(f"blocking eval ({country})"):
            eval_blocking(country, EVAL_N_QUERIES, pool_fraction=1.0, seed=config.RANDOM_SEED,
                          out=os.path.join(config.OUTPUT_DIR, f"blocking_eval_{country}.json"))
else:
    print("Skipped (RUN_BLOCKING_EVAL = False)")

## 3. Train (or load) the model

Trains on `SAMPLE_SIZE` S1 entities: blocking → features → XGBoost grid search → F₀.₅ threshold →
hard-negative round 2 (kept only if it improves validation F₀.₅). Saves the model and threshold to
`src/models/xgb_model.pkl`.

With `TRAIN_MODEL = False` the saved model is loaded instead. A model trained **before**
transliteration was added saw differently preprocessed text, so retrain rather than reuse it.

In [ ]:
from entity_resolution.pipeline import load_model, run_train

if TRAIN_MODEL:
    with timed("train"):
        model, threshold = run_train(sample_size=SAMPLE_SIZE, resume=RESUME)
else:
    # Refuses a model trained with different preprocessing/features than the current code.
    model, threshold = load_model()

print(f"Decision threshold: {threshold:.4f}")

## 4. Predict the full test set

Country by country (US, India, France): build the blockers over that country's full S2+S3
test pool, generate candidates, score them, apply singleton post-processing, then write both
output files. This is the long step; it saves each country's candidates and every scored shard
of 100K entities, so a restart repeats at most one shard.

To keep it running when the browser tab disconnects, you can instead run it in a terminal:

```bash
cd student_resource
nohup python code/business_entity_resolution/src/run_pipeline.py --mode full --sample-size 50000 > run.log 2>&1 &
tail -f run.log
```

It uses the same checkpoints, so notebook and terminal runs can pick up each other's work.

In [ ]:
from entity_resolution.pipeline import predict_test_by_country

with timed("test prediction"):
    matches, candidates = predict_test_by_country(model, threshold, resume=RESUME)

## 5. Validate the submission files

Runs the challenge's own validator. It must print `PASS` before you upload.

In [ ]:
import subprocess

validator = Path(config.BASE_DIR) / "utils" / "validate_submission.py"
result = subprocess.run(
    [sys.executable, str(validator),
     "--matching", os.path.join(config.OUTPUT_DIR, "matching_results.tsv"),
     "--candidate", os.path.join(config.OUTPUT_DIR, "candidate_pairs.tsv"),
     "--test-dir", os.path.join(config.BASE_DIR, "dataset", "test")],
    cwd=config.BASE_DIR, capture_output=True, text=True,
)
print(result.stdout, result.stderr)
if result.returncode != 0:
    raise RuntimeError("Validation FAILED — fix the issues above before uploading.")
print("Validation PASSED — output/matching_results.tsv is ready to upload.")

## 6. Results

In [ ]:
import pandas as pd
from entity_resolution.data import load_source

s1 = load_source(config.TEST_S1)[["entity_id", "country_norm"]]
res = s1.assign(
    n_matches=s1["entity_id"].map(lambda s: len(matches.get(s, []))),
    n_candidates=s1["entity_id"].map(lambda s: len(candidates.get(s, []))),
)

summary = res.groupby("country_norm").agg(
    entities=("entity_id", "size"),
    predicted_singletons=("n_matches", lambda x: (x == 0).mean()),
    avg_matches=("n_matches", "mean"),
    avg_matches_if_any=("n_matches", lambda x: x[x > 0].mean()),
    max_matches=("n_matches", "max"),
    avg_candidates=("n_candidates", "mean"),
)
summary.loc["ALL"] = [len(res), (res.n_matches == 0).mean(), res.n_matches.mean(),
                      res.n_matches[res.n_matches > 0].mean(), res.n_matches.max(),
                      res.n_candidates.mean()]
formats = {"entities": "{:,.0f}", "predicted_singletons": "{:.1%}", "avg_matches": "{:.2f}",
           "avg_matches_if_any": "{:.2f}", "max_matches": "{:.0f}", "avg_candidates": "{:.1f}"}
display(summary.apply(lambda col: col.map(formats[col.name].format)))
print("Training data for reference: 5.6% singletons, 3.67 matches per matched entity (max 11).")

print("\nMatches per entity (share of entities):")
print(res["n_matches"].clip(upper=12).value_counts(normalize=True).sort_index()
      .rename(lambda k: "12+" if k == 12 else k).map("{:.1%}".format).to_string())

In [ ]:
for name in ("matching_results.tsv", "candidate_pairs.tsv"):
    path = os.path.join(config.OUTPUT_DIR, name)
    print(f"{path}  ({os.path.getsize(path) / 2**20:,.1f} MB)")

print("\nmatching_results.tsv preview:")
display(pd.read_csv(os.path.join(config.OUTPUT_DIR, "matching_results.tsv"),
                    sep="\t", dtype=str, nrows=10, keep_default_na=False))

print("Stage timings:")
for label, sec in TIMINGS.items():
    print(f"  {label:28s} {sec / 60:7.1f} min")

## 7. Build the submission zip

Creates `student_resource/<TEAM_NAME>_submission.zip` in the structure the challenge README requires:

```
<team_name>_submission.zip
├── output/
│   ├── matching_results.tsv
│   └── candidate_pairs.tsv
├── code/
│   └── business_entity_resolution/
│       ├── src/
│       ├── README.md
│       └── requirements.txt      # pinned to the versions installed in this environment
└── Documentation_template.md
```

Before submitting, fill in the placeholders in `student_resource/Documentation_template.md`
(team name, members, date, the F₀.₅ score and the full-pool blocking tables). This cell warns
if any are left.

In [ ]:
from entity_resolution.submission import build_submission_zip

zip_path = build_submission_zip(TEAM_NAME)

## Next steps

- Upload `student_resource/output/matching_results.tsv` to the portal (the leaderboard file).
- Fill in the placeholders in `Documentation_template.md`, re-run the zip cell, and submit the zip.